# Tome 5, Chapter 5: The prime number theorem (Python companion)

Exercises first, solutions at the end. We count primes up to $10^9$ with a segmented sieve and compare with $\frac x{\log x}$ and $\operatorname{Li}(x)$ (Figure 5.1), watch the integral of Theorem 5.5 converge to $-1-\gamma$, test Theorem 5.7, the $n$-th prime, the progressions of Theorem 5.8, and finally use the prime number theorem to predict the cost of generating RSA primes. Requirements: numpy, mpmath.

*Before each solution, predict what the output will look like.*

## Exercises

**P5.1 (★★) Counting to a billion.** A plain sieve of Eratosthenes up to $10^9$ needs a gigabyte. Write a *segmented* sieve: sieve the blocks $[k\cdot10^7,(k+1)\cdot10^7)$ one at a time with the primes up to $\sqrt{10^9}$. Compute $\pi(10^k)$ for $k\le9$ and compare with $\frac x{\log x}$ and $\operatorname{Li}(x)$.

**P5.2 (★) Newman's integral.** Compute $\frac{\psi(X)}X$, $\int_1^X\frac{\psi(t)-t}{t^2}\,dt$ and $\sum_{n\le X}\frac{\Lambda(n)}n-\log X$ for $X=10^2,\dots,10^8$, and compare with Theorem 5.5 and Exercise 5.2. (The integral equals $\sum_{n\le X}\Lambda(n)(\frac1n-\frac1X)-\log X$: why?)

**P5.3 (★) The mean of $\mu$.** Compute $A(x)=\sum_{n\le x}\frac{\mu(n)}n$, $\frac{M(x)}x$ and $\int_1^x\frac{A(t)}t\,dt$ up to $10^7$ (Theorem 5.7 and its proof, Exercise 5.3).

**P5.4 (★) The $n$-th prime.** Compare $p_n$ with $n\log n$ and with Cipolla's $n(\log n+\log\log n-1)$ (Exercise 5.4).

**P5.5 (★★) Progressions.** For $q=3$ and $4$, compute $-\frac{L'}L(1,\chi)$ with mpmath and compare it with the difference between the two classes of $\sum\frac{\Lambda(n)}n$ (Exercise 5.8). Then watch $R_a(x)$ converge (Theorem 5.8(b)).

**P5.6 (★★) CS application: the cost of an RSA key.** To generate a $b$-bit prime, one draws random odd $b$-bit numbers until a Miller–Rabin test says ``prime''. By the prime number theorem a random number near $2^b$ is prime with probability about $\frac1{b\log2}$, and an odd one with probability $\frac2{b\log2}$. Predict the mean number of candidates, and measure it for $b=256,512,1024$.

## Solutions

### P5.1

In [ ]:
import math, time
import numpy as np
import mpmath as mp

def small_primes(n):
    s = np.ones(n + 1, dtype=bool); s[:2] = False
    for i in range(2, math.isqrt(n) + 1):
        if s[i]:
            s[i * i::i] = False
    return np.flatnonzero(s)

def segmented_count(X, checkpoints, seg=10**7):
    """Count primes up to X with a segmented sieve; memory O(sqrt(X) + seg). Returns {x: pi(x)} for x in checkpoints."""
    base = small_primes(math.isqrt(X) + 1)
    out, total, lo = {}, 0, 0
    cps = sorted(checkpoints)
    while lo <= X:
        hi = min(lo + seg, X + 1)                     # segment [lo, hi)
        s = np.ones(hi - lo, dtype=bool)
        if lo == 0:
            s[:2] = False
        for p in base:
            if p * p >= hi:
                break
            start = max(p * p, ((lo + p - 1) // p) * p)
            s[start - lo::p] = False
        idx = np.flatnonzero(s) + lo
        while cps and cps[0] < hi:
            out[cps[0]] = total + int(np.searchsorted(idx, cps[0], side='right'))
            cps.pop(0)
        total += len(idx)
        lo = hi
    return out

li2 = float(mp.li(2))
Li = lambda x: float(mp.li(x)) - li2
t0 = time.time()
X = 10**9
res = segmented_count(X, [10**k for k in range(1, 10)])
print(f"segmented sieve up to 1e9: {time.time() - t0:.1f} s")
print(f"{'x':>6} {'pi(x)':>11} {'x/log x':>11} {'Li(x)':>11} {'ratio1':>8} {'ratio2':>9} {'Li-pi':>6}")
for k in range(1, 10):
    x = 10**k; c = res[x]
    print(f"1e{k:<4} {c:11d} {x / math.log(x):11.0f} {Li(x):11.0f} {c * math.log(x) / x:8.4f} {c / Li(x):9.6f} {Li(x) - c:6.0f}")

Eight seconds and $O(\sqrt x+\text{block})$ memory for $\pi(10^9)=50\,847\,534$. The ratio $\frac{\pi(x)\log x}x$ falls very slowly ($1.054$ at $10^9$, like $1+\frac1{\log x}$, Exercise 5.1), while $\operatorname{Li}(x)$ overshoots by only $1700$, about $\frac{\sqrt x}{\log x}\cdot 1.1$. That size, $\sqrt x$ up to logarithms, is what the Riemann hypothesis predicts (Chapter 7). The segmented sieve is the standard way to sieve beyond memory; the record computations of $\pi(x)$ (beyond $10^{27}$) use instead the combinatorial Meissel–Lehmer method or analytic methods based on $\zeta$.

### P5.2

In [ ]:
N = 10**8
P = small_primes(N)
lam_n, lam_v = [], []
for p in P:
    p = int(p); lp = math.log(p); pk = p
    while pk <= N:
        lam_n.append(pk); lam_v.append(lp); pk *= p
order = np.argsort(lam_n)
lam_n = np.array(lam_n)[order]; lam_v = np.array(lam_v)[order]
cs_v = np.cumsum(lam_v); cs_w = np.cumsum(lam_v / lam_n)
gamma = float(mp.euler)
print(f"target: -1 - gamma = {-1 - gamma:.6f};  -gamma = {-gamma:.6f}")
for k in range(2, 9):
    X = 10**k; j = np.searchsorted(lam_n, X, side='right')
    psi, S = cs_v[j - 1], cs_w[j - 1]
    integral = S - psi / X - math.log(X)            # int_1^X (psi(t)-t)/t^2 dt
    print(f"X = 1e{k}: psi/X = {psi / X:.6f}, integral = {integral:+.6f}, sum Lambda/n - log X = {S - math.log(X):+.6f}")

The identity: $\int_1^X\frac{\psi(t)}{t^2}dt=\sum_{n\le X}\Lambda(n)\int_n^X\frac{dt}{t^2}=\sum_{n\le X}\Lambda(n)(\frac1n-\frac1X)$. The integral converges quickly to $-1-\gamma=-1.577216$ (six digits at $10^8$), much faster than $\frac{\psi(X)}X\to1$, because the integral averages the fluctuations of $\psi(t)-t$. And $\sum_{n\le X}\frac{\Lambda(n)}n-\log X$ approaches $-\gamma$ with oscillations of relative size $\frac{\psi(X)-X}X$.

### P5.3

In [ ]:
M = 10**7
mu = np.ones(M + 1, dtype=np.int64); mu[0] = 0
for p in P[P <= M]:
    mu[::p] *= -1
    mu[::p * p] = 0
ratio = np.zeros(M + 1); ratio[1:] = mu[1:] / np.arange(1, M + 1)
A = np.cumsum(ratio); Mx = np.cumsum(mu)
n = np.arange(1, M)
I = np.cumsum(A[1:M] * np.log((n + 1) / n))         # int_1^x A(t)/t dt at integers
for k in range(1, 8):
    x = 10**k
    print(f"x = 1e{k}: A(x) = {A[x]:+.6f}, M(x)/x = {Mx[x] / x:+.6f}, int_1^x A(t)/t dt = {I[x - 2] if x > 1 else 0:.6f}")

$A(x)$ changes sign and decreases in size roughly like $\frac1{\sqrt x}$ at these heights, $\frac{M(x)}x$ follows it closely, and $\int_1^x\frac{A(t)}t\,dt$ converges to $1=g(0)$, the value given by Newman's theorem in the proof of Theorem 5.7. Note that $\frac{M(x)}x\approx A(x)$ here: by Abel summation $M(x)=xA(x)-\int_1^xA$, and the integral is small.

### P5.4

In [ ]:
for n in (10**3, 10**4, 10**5, 10**6, 10**7, 5 * 10**6):
    if n > len(P): continue
    p = int(P[n - 1]); L = math.log(n)
    print(f"n = {n:>8}: p_n = {p:>10}, p_n/(n log n) = {p / (n * L):.4f}, Cipolla p_n/(n(log n + log log n - 1)) = {p / (n * (L + math.log(L) - 1)):.5f}")

$\frac{p_n}{n\log n}$ decreases to $1$ extremely slowly ($1.115$ at $n=5\cdot10^6$), while Cipolla's second-order formula is within $0.3\%$: the next correction is of order $\frac{\log\log n}{\log^2n}$ relative to the main term.

### P5.5

In [ ]:
def lam_class_sum(q, a, X):
    j = np.searchsorted(lam_n, X, side='right')
    sel = lam_n[:j] % q == a
    return (lam_v[:j][sel] / lam_n[:j][sel]).sum()
mp.mp.dps = 20
def LprimeL(q, chi):
    f = lambda s: sum(c * mp.zeta(s, mp.mpf(a) / q) for a, c in chi.items()) * mp.mpf(q)**(-s)
    s0 = 1 + mp.mpf('1e-12')
    return float(mp.diff(f, s0) / f(s0))
for q, chi in ((4, {1: 1, 3: -1}), (3, {1: 1, 2: -1})):
    print(f"q = {q}: -L'/L(1,chi) = {-LprimeL(q, chi):+.6f}")
    for k in (5, 6, 7, 8):
        X = 10**k
        d = lam_class_sum(q, 1, X) - lam_class_sum(q, q - 1, X)
        print(f"   X = 1e{k}: sum_(n=1) - sum_(n=-1) of Lambda(n)/n = {d:+.6f}")
    for a in (1, q - 1):
        vals = []
        for k in (6, 7, 8):
            X = 10**k; j = np.searchsorted(P, X, side='right'); Q = P[:j]
            sel = Q % q == a
            vals.append((np.log(Q[sel]) / Q[sel]).sum() - math.log(X) / 2)
        print(f"   R_{a}(x) at 1e6, 1e7, 1e8: " + ", ".join(f"{v:+.4f}" for v in vals))

The class differences converge to $-\frac{L'}L(1,\chi)$: $-0.245610$ for $q=4$ and $-0.368282$ for $q=3$, with errors of a few units in the fifth decimal at $10^8$. The $R_a(x)$ settle too, which is Theorem 5.8(b): for $q=3$ the limits are near $-1.188$ and $-0.511$, as in Figure 4.1. Their difference (about $-0.676$) splits into $-0.368$ from $\frac{L'}L$ and about $-0.308$ from the prime powers $p^k\equiv1$ that $R_a$ leaves out.

### P5.6

In [ ]:
import random
random.seed(5)
def is_probable_prime(n, rounds=20):
    if n < 2: return False
    for p in (2, 3, 5, 7, 11, 13, 17, 19, 23, 29):
        if n % p == 0: return n == p
    d, r = n - 1, 0
    while d % 2 == 0: d //= 2; r += 1
    for _ in range(rounds):
        x = pow(random.randrange(2, n - 1), d, n)
        if x in (1, n - 1): continue
        for _ in range(r - 1):
            x = x * x % n
            if x == n - 1: break
        else:
            return False
    return True
def trials_to_prime(bits):
    t = 0
    while True:
        t += 1
        c = random.getrandbits(bits) | (1 << (bits - 1)) | 1     # odd, exactly `bits` bits
        if is_probable_prime(c):
            return t
for bits in (256, 512, 1024):
    runs = [trials_to_prime(bits) for _ in range(60 if bits < 1024 else 30)]
    print(f"{bits} bits: mean odd candidates = {np.mean(runs):6.1f}, PNT prediction (bits log 2)/2 = {bits * math.log(2) / 2:6.1f}")

Prediction: an odd $b$-bit number is prime with probability $\approx\frac2{b\log 2}$, so the number of trials is geometric with mean $\frac{b\log2}2$: about $89$, $177$ and $355$. The measured means agree up to the large spread of a geometric law: its standard deviation equals its mean, so a mean over $30$ runs is uncertain by about $18\%$, and the $1024$-bit value is about one such standard error low. In practice, generators first discard the candidates with a prime factor up to some $y$ (trial division, as in Chapter 1's notebook), which removes most composites cheaply. All primes survive, and a fraction about $\frac{e^{-\gamma}}{\log y}$ of all integers does (Mertens's third theorem), so a survivor is prime with probability about $\frac{e^\gamma\log y}{b\log2}$, and the number of expensive Miller–Rabin tests drops to about $\frac{e^{-\gamma}b\log2}{\log y}$: for $b=1024$ and $y=10^4$, about $43$ instead of $355$.